# Tutorial 1: Covariance & Variogram Models

**Corresponds to MATLAB `MODELSLIBtutorial.m`**

This notebook demonstrates:
1. All available covariance models plotted as C(h) vs h
2. Corresponding variograms γ(h) = C(0) − C(h)
3. Nested (additive) models

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from pybme import (
    exponential_cov, gaussian_cov, spherical_cov,
    matern_cov, nugget_cov, hole_cos_cov, eval_cov,
)

## 1. Define Covariance Models

PyBME provides six covariance model families. Each is parameterized by a **sill** (variance at lag 0) and a **range** (correlation length).

In [ ]:
h = np.linspace(0, 1.5, 300)

models = [
    ("Nugget",       "nugget",      [1.0]),
    ("Exponential",  "exponential", [1.0, 1.0]),
    ("Spherical",    "spherical",   [1.0, 1.0]),
    ("Gaussian",     "gaussian",    [1.0, 1.0]),
    ("Matérn ν=1.5", "matern",      [1.0, 1.0, 1.5]),
    ("Hole-Cosine",  "hole_cos",    [1.0, 0.5]),
]

print(f"{'Model':20s}  {'C(0)':>8s}  {'C(1.0)':>8s}")
print("-" * 42)
for name, model, params in models:
    c0 = eval_cov(0.0, model, params)
    c1 = eval_cov(1.0, model, params)
    print(f"{name:20s}  {c0:8.4f}  {c1:8.4f}")

## 2. Covariance Functions C(h)

Plot each model's covariance function as a function of lag distance h.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (name, model, params) in zip(axes.ravel(), models):
    c = eval_cov(h, model, params)
    ax.plot(h, c, 'b-', lw=1.5)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('h')
    ax.set_ylabel('C(h)')
    ax.set_ylim(bottom=-0.2)
    ax.grid(True, alpha=0.3)
fig.suptitle('Covariance Models C(h)', fontsize=13)
fig.tight_layout()
plt.show()

## 3. Variogram Models γ(h) = C(0) − C(h)

The variogram is the complementary representation. It equals zero at h=0 and approaches the sill.

In [ ]:
fig2, axes2 = plt.subplots(2, 3, figsize=(12, 7))
for ax, (name, model, params) in zip(axes2.ravel(), models):
    c = eval_cov(h, model, params)
    gamma = eval_cov(0.0, model, params) - c
    ax.plot(h, gamma, 'r-', lw=1.5)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('h')
    ax.set_ylabel('γ(h)')
    ax.grid(True, alpha=0.3)
fig2.suptitle('Variogram Models γ(h) = C(0) − C(h)', fontsize=13)
fig2.tight_layout()
plt.show()

## 4. Nested (Additive) Variogram Model

Nested models combine multiple basic models. Here: nugget(0.2) + spherical(0.4, 0.3) + spherical(0.4, 1.2).

In [ ]:
nest_model = ["nugget", "spherical", "spherical"]
nest_params = [[0.2], [0.4, 0.3], [0.4, 1.2]]

c_nest = eval_cov(h, nest_model, nest_params)
gamma_nest = eval_cov(0.0, nest_model, nest_params) - c_nest

print(f"Nested model C(0) = {eval_cov(0.0, nest_model, nest_params):.4f}")

fig3, ax3 = plt.subplots(figsize=(8, 4))
ax3.plot(h, gamma_nest, 'k-', lw=2,
         label='Nested: nug(0.2) + sph(0.4,0.3) + sph(0.4,1.2)')
ax3.set_xlabel('h')
ax3.set_ylabel('γ(h)')
ax3.set_title('Nested Variogram Model')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
fig3.tight_layout()
plt.show()

## 5. Space-Time Covariance Models

MATLAB BMElib supports two flavours of S/T covariance:

- **Separable**: `'gaussianC/exponentialC'` → $C(r,t) = \sigma^2 \, C_s(r) \, C_t(t)$
- **Non-separable**: `'gaussianCST'` → uses a combined S/T metric $d_{st} = r + k\,t$

The sill (maximum covariance) is always at spatial lag $r=0$ and temporal lag $t=0$.

In [ ]:
from pybme import eval_cov_st

# Lag grids
r = np.linspace(0, 3.0, 200)   # spatial lags
t = np.linspace(0, 6.0, 200)   # temporal lags

# ── Separable model: exponential(space) / exponential(time) ──
sill = 1.0
range_s, range_t = 1.0, 2.0

# 1D slices
C_r = eval_cov_st(r, 0.0, 'exponential', [1.0, range_s],
                   'exponential', [1.0, range_t], sill=sill)
C_t = eval_cov_st(0.0, t, 'exponential', [1.0, range_s],
                   'exponential', [1.0, range_t], sill=sill)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(r, C_r, 'b-', lw=2)
ax1.set_xlabel('Spatial lag r')
ax1.set_ylabel('C(r, t=0)')
ax1.set_title('Separable S/T Covariance — Spatial Slice (t = 0)')
ax1.axhline(sill, color='gray', ls='--', lw=0.8, label=f'sill = {sill}')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(t, C_t, 'r-', lw=2)
ax2.set_xlabel('Temporal lag t')
ax2.set_ylabel('C(r=0, t)')
ax2.set_title('Separable S/T Covariance — Temporal Slice (r = 0)')
ax2.axhline(sill, color='gray', ls='--', lw=0.8, label=f'sill = {sill}')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.suptitle(f'Separable:  C(r,t) = {sill} · exp_s(r, a_s={range_s}) · exp_t(t, a_t={range_t})',
             fontsize=11)
fig.tight_layout()
plt.show()

print(f"Sill check: C(0,0) = {eval_cov_st(0, 0, 'exponential', [1,range_s], 'exponential', [1,range_t], sill=sill):.4f}")

## 6. S/T Covariance Surface  C(r, t)

A 2-D color map of the separable covariance — spatial lag on the x-axis, temporal lag on the y-axis. The sill (maximum) is at the origin (r=0, t=0).

In [ ]:
# 2-D surface: C(r, t) with spatial lag on x-axis, temporal lag on y-axis
rg = np.linspace(0, 3.0, 100)
tg = np.linspace(0, 6.0, 100)
R, T = np.meshgrid(rg, tg)

C_sep = eval_cov_st(R, T, 'exponential', [1.0, range_s],
                     'exponential', [1.0, range_t], sill=sill)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(R, T, C_sep, cmap='viridis', shading='auto')
ax.set_xlabel('Spatial lag r', fontsize=11)
ax.set_ylabel('Temporal lag t', fontsize=11)
ax.set_title('Separable S/T Covariance  C(r, t)\n'
             f'exp/exp  sill={sill}, a_s={range_s}, a_t={range_t}', fontsize=12)
cb = plt.colorbar(im, ax=ax, label='C(r, t)')
ax.plot(0, 0, 'w*', ms=14, zorder=5)
ax.annotate(f'sill = {sill}', xy=(0, 0), xytext=(0.3, 0.5),
            fontsize=10, color='white',
            arrowprops=dict(arrowstyle='->', color='white'))
fig.tight_layout()
plt.show()

## 7. Non-Separable S/T Covariance

A non-separable model combines spatial and temporal lags into a single metric:  
$d_{st} = r + k \cdot t$, then applies a standard covariance to that combined distance.

This means contours of equal covariance are **straight lines** (not axis-aligned rectangles like separable models).

In [ ]:
# Non-separable Gaussian S/T model:  d_st = r + k*t
sill_ns = 1.0
range_st = 2.0
k = 0.5  # space-time metric

# 1D slices
C_r_ns = eval_cov_st(r, 0.0, 'gaussian_st', [sill_ns, range_st, k])
C_t_ns = eval_cov_st(0.0, t, 'gaussian_st', [sill_ns, range_st, k])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(r, C_r_ns, 'b-', lw=2)
ax1.set_xlabel('Spatial lag r')
ax1.set_ylabel('C(r, t=0)')
ax1.set_title('Non-Separable Gaussian ST — Spatial Slice')
ax1.axhline(sill_ns, color='gray', ls='--', lw=0.8, label=f'sill = {sill_ns}')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(t, C_t_ns, 'r-', lw=2)
ax2.set_xlabel('Temporal lag t')
ax2.set_ylabel('C(r=0, t)')
ax2.set_title('Non-Separable Gaussian ST — Temporal Slice')
ax2.axhline(sill_ns, color='gray', ls='--', lw=0.8, label=f'sill = {sill_ns}')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.suptitle(f'Non-separable:  d_st = r + {k}·t,  Gaussian(sill={sill_ns}, range={range_st})',
             fontsize=11)
fig.tight_layout()
plt.show()

print(f"Sill check: C(0,0) = {eval_cov_st(0, 0, 'gaussian_st', [sill_ns, range_st, k]):.4f}")

## 8. Separable vs Non-Separable — Side by Side

Compare the 2-D covariance surfaces. Notice how:
- **Separable** contours are axis-aligned rectangles (product of independent spatial and temporal functions)
- **Non-separable** contours are diagonal lines (spatial and temporal lags trade off via the metric $d_{st} = r + k \, t$)

In [ ]:
# Side-by-side comparison
C_nonsep = eval_cov_st(R, T, 'gaussian_st', [sill_ns, range_st, k])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Separable
im1 = ax1.pcolormesh(R, T, C_sep, cmap='viridis', shading='auto',
                      vmin=0, vmax=1)
cs1 = ax1.contour(R, T, C_sep, levels=[0.05, 0.1, 0.2, 0.4, 0.6, 0.8],
                   colors='w', linewidths=0.8)
ax1.clabel(cs1, fontsize=8, fmt='%.2f')
ax1.plot(0, 0, 'w*', ms=14, zorder=5)
ax1.set_xlabel('Spatial lag r', fontsize=11)
ax1.set_ylabel('Temporal lag t', fontsize=11)
ax1.set_title('Separable: exp(r)/exp(t)', fontsize=11)
plt.colorbar(im1, ax=ax1, label='C(r,t)')

# Non-separable
im2 = ax2.pcolormesh(R, T, C_nonsep, cmap='viridis', shading='auto',
                      vmin=0, vmax=1)
cs2 = ax2.contour(R, T, C_nonsep, levels=[0.05, 0.1, 0.2, 0.4, 0.6, 0.8],
                   colors='w', linewidths=0.8)
ax2.clabel(cs2, fontsize=8, fmt='%.2f')
ax2.plot(0, 0, 'w*', ms=14, zorder=5)
ax2.set_xlabel('Spatial lag r', fontsize=11)
ax2.set_ylabel('Temporal lag t', fontsize=11)
ax2.set_title(f'Non-separable: gaussian_st (k={k})', fontsize=11)
plt.colorbar(im2, ax=ax2, label='C(r,t)')

fig.suptitle('Space-Time Covariance C(r,t)  —  sill at (0, 0)', fontsize=13)
fig.tight_layout()
plt.show()